# Nuclear Geometry and Characterization of the Quark-Gluon Plasma

<div>
<img src="https://raw.githubusercontent.com/illinois-ipaml/MachineLearningForPhysics/main/img/Project_NuclearGeometryQCD-Figure.jpg" width=400></img>
</div>

## <span style="color:Orange">Overview</span>

Collisions between two nuclei at ultra-relativistic energies create the quark-gluon plasma (QGP), a novel state of matter. In [Project 01](https://illinois-ipaml.github.io/MachineLearningForPhysics/_sources/projects/Project_01_NuclearGeometryQGP.html) you learned how the geometry of the overlap region between two colliding nuclei is characterized by eccentricities $\varepsilon_n$, and how these fluctuate from event to event.

The QGP converts this initial spatial anisotropy into an anisotropy in the momenta of the produced particles, which is measured as the Fourier coefficients $v_n$ of their azimuthal distribution ("anisotropic flow"). To a good approximation, $v_n \propto \varepsilon_n$ for $n = 2, 3$, with a proportionality constant that depends on the properties of the QGP, such as its shear viscosity. In this project you will:

* simulate events with anisotropic flow and measure $v_n$ with two-particle correlations;
* investigate whether two-particle correlations are sensitive to correlations between the symmetry-plane angles $\Psi_n$;
* use a Monte Carlo model and the **symmetric cumulant** observable to study correlations between the *magnitudes* of $v_2$ and $v_3$, with bootstrap uncertainties; and
* compute the corresponding correlation between $\varepsilon_2$ and $\varepsilon_3$ from the Monte Carlo Glauber events of Project 01.

## <span style="color:Orange">Data Sources</span>

The data are the same Monte Carlo Glauber [[1]](https://doi.org/10.1103/PhysRevC.97.054910) simulations of Pb+Pb collisions used in Project 01. Each file contains 10,000 collisions and gives the transverse positions $(x, y)$, in fm, of every nucleon that participated in a collision. They are used in Question 07.

* $b = 0$ fm: https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/NuclearGeometryQGC/PbPb_b0_10kevents.csv (98 MB)
* $b = 6$ fm: https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/NuclearGeometryQGC/PbPb_b6_10kevents.csv (59 MB)

⚠️ **Data-quality issue:** the program that produced these files wrote its progress messages into them. At the start of every 100th event, the event number is prefixed with text such as `1% done`, giving lines like `1% done100, -3.1954, 1.69976` (99 lines per file). A plain `pd.read_csv` therefore reads the `event` column as text, and grouping by it creates 99 spurious one-nucleon "events". The header also has stray spaces. The `load_glauber` function below fixes both problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_URL = "https://courses.physics.illinois.edu/phys498mlp/sp2025/data/projects/NuclearGeometryQGC/"

def load_glauber(filename):
    """Read a Glauber file, removing the stray progress messages from the event column."""
    df = pd.read_csv(BASE_URL + filename, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    df["event"] = df["event"].astype(str).str.replace(r"^\d+% done", "", regex=True).astype(int)
    return df

glauber = {b: load_glauber(f"PbPb_b{b}_10kevents.csv") for b in (0, 6)}
for b, df in glauber.items():
    print(f"b = {b} fm: {df['event'].nunique()} events, {len(df)} participating nucleons")

## <span style="color:Orange">Background</span>

As in Project 01, the shape of the overlap region can be characterized by the eccentricities

$$
\Large \varepsilon_n = \frac{\sqrt{\langle r^2\cos(n\phi)\rangle^2 + \langle r^2\sin(n\phi)\rangle^2}}{\langle r^2 \rangle}
$$

(Project 01 looked at $n = 2$ and $n = 3$). This $r^2$ weighting follows Ref. [[2]](https://doi.org/10.1103/PhysRevC.81.054905); some authors use $r^n$ instead for $n \geq 3$. It is useful to think of each harmonic as a two-dimensional vector $\vec{\varepsilon}_n = \varepsilon_n e^{in\Psi_n}$, whose magnitude is $\varepsilon_n$ and whose direction is set by the phase $\Psi_n$, the orientation of the $n$-th order symmetry plane. The produced particles inherit this structure: in each event, their azimuthal angles $\phi$ follow

$$
\frac{dN}{d\phi} \propto 1 + \sum_{n=1}^{\infty} 2 v_n \cos\left(n(\phi - \Psi_n)\right).
$$

We can ask two separate questions about how the harmonics are related: are the **angles** $\Psi_2$ and $\Psi_3$ correlated (Questions 02–03), and are the **magnitudes** $v_2$ and $v_3$ correlated (Questions 04–07)?

## <span style="color:Orange">Questions</span>

Answer each question in a markdown cell directly below it (add code cells where needed). Make sure to cite any sources you use. Your notebook should run from top to bottom without errors. Fix the random seed (e.g. `rng = np.random.default_rng(42)`) so that your results are reproducible.

### <span style="color:LightGreen">Question 01</span>

(a) Plot the shapes described by each single harmonic for $n = 1$ to $5$, one plot for each $n$. Use polar plots of $r(\phi) = 1 + \varepsilon_n \cos(n\phi)$. For example, for $n = 1$, $r(\phi) = 1 + \varepsilon_1\cos\phi$. Use $\varepsilon_n = 0.1$, and also a larger value such as 0.3 to make the shapes easier to see. You may use Python or any other tool, such as Desmos.

(b) Read about the QGP in Ref. [[3]](https://doi.org/10.1126/science.1215901) (available through the university library) or the freely available review in Ref. [[4]](https://arxiv.org/abs/1802.04801). Write a short paragraph explaining the connection between the collision geometry, the measured Fourier coefficients $v_n$, and the properties of the QGP.

### <span style="color:LightGreen">Question 02</span>

Whether the various symmetry-plane angles $\Psi_n$ are correlated with each other (for example, is $\Psi_2$ correlated with $\Psi_3$?) is an interesting physics question. Generate events containing 50 particles each (1,000 events) whose azimuthal angles $\phi$ are drawn from

$$
\frac{dN}{d\phi} \propto 1 + 2v_2\cos\left(2(\phi-\Psi_2)\right) + 2v_3\cos\left(3(\phi-\Psi_3)\right),
$$

with $v_2 = 0.2$ and $v_3 = 0.1$ in every event, for two cases:

* **Case 1:** $\Psi_2 = \Psi_3$, with the common angle drawn uniformly in $[0, 2\pi)$ independently for every event.
* **Case 2:** $\Psi_2$ and $\Psi_3$ drawn uniformly and independently of each other in every event.

(*Hint:* accept–reject sampling works well here.)

The symmetry-plane angles are not observable in an experiment, but the angle differences between particles are. For every pair of particles $i \neq j$ *in the same event*, compute $\Delta\phi = \phi_i - \phi_j$, and histogram the pairs from all events. Averaging over the random $\Psi_n$, the expected pair distribution is

$$
\frac{dN^{\rm pairs}}{d\Delta\phi} \propto 1 + \sum_{n=1}^{\infty} 2v_n^2 \cos(n\Delta\phi).
$$

Show this for the $n = 2$ term. Then perform a least-squares fit of your histograms for Case 1 and Case 2 with a function that includes both the $n = 2$ and $n = 3$ terms, $A\left[1 + 2v_2^2\cos(2\Delta\phi) + 2v_3^2\cos(3\Delta\phi)\right]$, and report $v_2$ and $v_3$ with their uncertainties.

### <span style="color:LightGreen">Question 03</span>

Do you get consistent values of $v_2$ and $v_3$ for Case 1 and Case 2? Explain what you see, and what it implies about the information contained in two-particle correlations. What kind of measurement *would* be sensitive to a correlation between $\Psi_2$ and $\Psi_3$? (*Hint:* think about correlations of more than two particles that combine different harmonics. Ref. [[7]](https://doi.org/10.1103/PhysRevC.90.024905) describes measurements of such "event-plane correlations".)

### <span style="color:LightGreen">Question 04</span>

In each event, $v_2$ and $v_3$ are approximately proportional to the corresponding $\varepsilon_2$ and $\varepsilon_3$, which fluctuate from event to event. Write a Monte Carlo that generates values of $(v_2, v_3)$ for many events (say 10,000) from a bivariate Gaussian distribution with five parameters: the means $\langle v_2 \rangle$ and $\langle v_3 \rangle$, the standard deviations $\sigma_{v_2}$ and $\sigma_{v_3}$, and the correlation coefficient $\rho$ between $v_2$ and $v_3$.

*Hint:* for $\rho = \pm 1$ the covariance matrix is singular. Some samplers (e.g. those based on a Cholesky decomposition) will fail. Generating $v_3$ from two independent standard normals $z_1, z_2$ as $v_3 = \langle v_3\rangle + \sigma_{v_3}\left(\rho z_1 + \sqrt{1-\rho^2}\, z_2\right)$, with $v_2 = \langle v_2\rangle + \sigma_{v_2} z_1$, works for any $\rho$.

### <span style="color:LightGreen">Question 05</span>

Verify your Monte Carlo using the parameters

* $\langle v_2 \rangle = 0.25$, $\sigma_{v_2} = 0.05$
* $\langle v_3 \rangle = 0.18$, $\sigma_{v_3} = 0.05$
* $\rho = -1.0$, $0.0$, $0.5$ and $1.0$

Make a scatter plot of $v_2$ versus $v_3$ for each value of $\rho$. Check that the sample means, standard deviations and correlation coefficient agree with the inputs.

One way to look for correlations between the *magnitudes* of flow harmonics is the **symmetric cumulant** (SC) [[5]](https://doi.org/10.1103/PhysRevC.89.064904). It measures the correlation between the squared flow coefficients of two different orders $m$ and $n$:

$$
SC(m,n) = \langle\langle \cos(m\phi_i + n\phi_j - m\phi_k - n\phi_l) \rangle\rangle - \langle\langle \cos m(\phi_i - \phi_j) \rangle\rangle \langle\langle \cos n(\phi_i - \phi_j) \rangle\rangle = \langle v_m^2 v_n^2\rangle - \langle v_m^2\rangle \langle v_n^2\rangle,
$$

where $\langle\langle \cdots \rangle\rangle$ denotes an average over all combinations of distinct particles $i, j, k, l$ within an event, and then over events. Unlike the two-particle correlations of Question 02, $SC(m,n)$ does not depend on the symmetry-plane angles. ALICE measured $SC(3,2)$ to be negative and $SC(4,2)$ to be positive in Pb+Pb collisions [[6]](https://doi.org/10.1103/PhysRevLett.117.182301).

### <span style="color:LightGreen">Question 06</span>

Calculating symmetric cumulants directly from particle angles is time consuming, because of the four-particle correlation term. In practice, people use a technique based on "Q-vectors" that avoids looping over all combinations [[5]](https://doi.org/10.1103/PhysRevC.89.064904). With the Monte Carlo from Question 04, however, you can calculate $SC(3,2) = \langle v_2^2 v_3^2\rangle - \langle v_2^2\rangle \langle v_3^2\rangle$ directly from the generated $v_2$ and $v_3$ values.

* Do this for each value of $\rho$ in Question 05, and use a **bootstrap** to estimate the uncertainties.
* Plot $SC(3,2)$ as a function of $\rho$. Also try a finer grid of $\rho$ values.
* For a bivariate Gaussian, derive an analytic expression for $SC(3,2)$ in terms of the five parameters. (*Hint:* write $v_n = \langle v_n\rangle + \delta_n$, and use the fact that for zero-mean jointly Gaussian variables, $\langle\delta_2^2\delta_3^2\rangle = \sigma_{v_2}^2\sigma_{v_3}^2(1 + 2\rho^2)$.) Overlay it on your plot. Does your Monte Carlo agree?

### <span style="color:LightGreen">Question 07</span>

**Connecting to the collision geometry.** If $v_n \propto \varepsilon_n$, the sign of $SC(3,2)$ should reflect the correlation between $\varepsilon_2^2$ and $\varepsilon_3^2$ in the initial state. Using the Monte Carlo Glauber events (loaded with `load_glauber` above), calculate $\varepsilon_2$ and $\varepsilon_3$ for every event, centering each event so that $\langle x\rangle = \langle y\rangle = 0$ as in Project 01. Then, for $b = 0$ fm and $b = 6$ fm:

* Make a scatter plot (or 2D histogram) of $\varepsilon_3$ versus $\varepsilon_2$, and compute their correlation coefficient.
* Compute the normalized symmetric cumulant of the eccentricities,
  $$NSC_\varepsilon(3,2) = \frac{\langle \varepsilon_2^2\varepsilon_3^2\rangle - \langle\varepsilon_2^2\rangle\langle\varepsilon_3^2\rangle}{\langle\varepsilon_2^2\rangle\langle\varepsilon_3^2\rangle},$$
  with a bootstrap uncertainty.

What signs do you find, and how do they compare with the ALICE measurement of $SC(3,2)$ [[6]](https://doi.org/10.1103/PhysRevLett.117.182301)? Is a bivariate Gaussian, the model of Question 04, a good description of the joint distribution of $(\varepsilon_2, \varepsilon_3)$? Explain why or why not. You could compare it with a kernel density estimate of the joint distribution.

## <span style="color:Orange">References</span>

__[<span style="color:Red">1</span>]__ C. Loizides, J. Kamin, D. d'Enterria, "Improved Monte Carlo Glauber predictions at present and future nuclear colliders," *Phys. Rev. C* **97**, 054910 (2018); Erratum *Phys. Rev. C* **99**, 019901 (2019). [doi:10.1103/PhysRevC.97.054910](https://doi.org/10.1103/PhysRevC.97.054910), [arXiv:1710.07098](https://arxiv.org/abs/1710.07098)

__[<span style="color:Red">2</span>]__ B. Alver, G. Roland, "Collision-geometry fluctuations and triangular flow in heavy-ion collisions," *Phys. Rev. C* **81**, 054905 (2010); Erratum *Phys. Rev. C* **82**, 039903 (2010). [doi:10.1103/PhysRevC.81.054905](https://doi.org/10.1103/PhysRevC.81.054905), [arXiv:1003.0194](https://arxiv.org/abs/1003.0194)

__[<span style="color:Red">3</span>]__ B. V. Jacak, B. Müller, "The Exploration of Hot Nuclear Matter," *Science* **337**, 310 (2012). [doi:10.1126/science.1215901](https://doi.org/10.1126/science.1215901)

__[<span style="color:Red">4</span>]__ W. Busza, K. Rajagopal, W. van der Schee, "Heavy Ion Collisions: The Big Picture, and the Big Questions," *Ann. Rev. Nucl. Part. Sci.* **68**, 339 (2018). [doi:10.1146/annurev-nucl-101917-020852](https://doi.org/10.1146/annurev-nucl-101917-020852), [arXiv:1802.04801](https://arxiv.org/abs/1802.04801)

__[<span style="color:Red">5</span>]__ A. Bilandzic, C. H. Christensen, K. Gulbrandsen, A. Hansen, Y. Zhou, "Generic framework for anisotropic flow analyses with multiparticle azimuthal correlations," *Phys. Rev. C* **89**, 064904 (2014). [doi:10.1103/PhysRevC.89.064904](https://doi.org/10.1103/PhysRevC.89.064904), [arXiv:1312.3572](https://arxiv.org/abs/1312.3572)

__[<span style="color:Red">6</span>]__ ALICE Collaboration, "Correlated event-by-event fluctuations of flow harmonics in Pb-Pb collisions at √s<sub>NN</sub> = 2.76 TeV," *Phys. Rev. Lett.* **117**, 182301 (2016). [doi:10.1103/PhysRevLett.117.182301](https://doi.org/10.1103/PhysRevLett.117.182301), [arXiv:1604.07663](https://arxiv.org/abs/1604.07663)

__[<span style="color:Red">7</span>]__ ATLAS Collaboration, "Measurement of event-plane correlations in √s<sub>NN</sub> = 2.76 TeV lead-lead collisions with the ATLAS detector," *Phys. Rev. C* **90**, 024905 (2014). [doi:10.1103/PhysRevC.90.024905](https://doi.org/10.1103/PhysRevC.90.024905), [arXiv:1403.0489](https://arxiv.org/abs/1403.0489)

## <span style="color:Orange">Acknowledgements</span>

* Initial version: Anne Sickles
* Revised: 2026

© Copyright 2026